In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DATA_DIR = Path(os.environ["DATA_DIR"]).expanduser().resolve()

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

print(DATA_DIR)


G:\Mon Drive\Quantitative_Trading_and_Price_Impact_2026


In [2]:
year = "2019"
bin_sample_path = f"{DATA_DIR}/Data/binSamples/"
fill_sample_path = f"{DATA_DIR}/Data/fillSamples/"
result_path = f"{DATA_DIR}/inter_results/"
os.makedirs(result_path, exist_ok=True)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Data Preparation

1) Baseline : In-sample data on one month (January) and out-of-sample data on the next month (February) over 20 stocks.

In [ ]:
year = "2019"
in_sample_month = "01"
out_of_sample_month = "02"

in_sample_bin_df = pd.read_csv(f"{bin_sample_path}bin{year}{in_sample_month}.csv")
out_of_sample_bin_df = pd.read_csv(f"{bin_sample_path}bin{year}{out_of_sample_month}.csv")

stocks = ['A', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABMD', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADS', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AGN']

in_sample_bin_df = in_sample_bin_df[in_sample_bin_df['stock'].isin(stocks)]
out_of_sample_bin_df = out_of_sample_bin_df[out_of_sample_bin_df['stock'].isin(stocks)]



<StringArray>
['AAPL']
Length: 1, dtype: str

# Impact Model Fitting

In [27]:
traded_volume_df = in_sample_bin_df[["stock", "date","trade", "time"]].pivot(index=["stock", "date"], columns=["time"])["trade"].fillna(0).astype(int)

px_df = in_sample_bin_df[["stock", "date", "midEnd", "time"]].pivot(index=["stock", "date"], columns=["time"])["midEnd"].ffill(axis="columns").bfill(axis="columns")

stock_info_df = pd.DataFrame({"px_vol" : px_df.pct_change(1, axis="columns").std(axis="columns"),
        "volume" : traded_volume_df.abs().sum(axis="columns"),}).reset_index()

In [31]:
scaling_df = (
    stock_info_df
    .groupby("stock")[["px_vol", "volume"]]
    .mean()
    .reset_index()
)

In [34]:
def impact_state(traded_volume_df, monthly_scaling_factor, half_life, model_type):

    space_kernels = {
        "linear" : lambda x : x,
        "sqrt" : lambda x : np.sign(x) * np.sqrt(np.abs(x)),
    }

    beta = np.log(2) / (half_life / 10)
    decay_factor = np.exp(-beta)
    pre_ewm = traded_volume_df.copy()
    pre_ewm = pre_ewm.divide(monthly_scaling_factor["volume"], axis="rows")
    pre_ewm = space_kernels[model_type](pre_ewm)
    pre_ewm = pre_ewm.multiply(monthly_scaling_factor["px_vol"], axis="rows")
    pre_ewm.iloc[:, 1:] /= (1 - decay_factor)
    cum_impact = cum_impact = pre_ewm.T.ewm(alpha=1-decay_factor, adjust=False).mean().T
    return cum_impact

In [35]:
def impact_state(traded_volume_df, monthly_scaling_factor, half_life, model_type):

    space_kernels = {
        "linear" : lambda x : x,
        "sqrt" : lambda x : np.sign(x) * np.sqrt(np.abs(x)),
    }

    beta = np.log(2) / (half_life / 10)
    decay_factor = np.exp(-beta)
    pre_ewm = traded_volume_df.copy()
    pre_ewm = pre_ewm.divide(monthly_scaling_factor["volume"], axis="rows")
    pre_ewm = space_kernels[model_type](pre_ewm)
    pre_ewm = pre_ewm.multiply(monthly_scaling_factor["px_vol"], axis="rows")
    pre_ewm.iloc[:, 1:] /= (1 - decay_factor)
    cum_impact = cum_impact = pre_ewm.T.ewm(alpha=1-decay_factor, adjust=False).mean().T
    return cum_impact

In [39]:
monthly_scaling_factor = scaling_df.set_index("stock").loc[traded_volume_df.index.get_level_values("stock")].reset_index(drop=True)
monthly_scaling_factor.index = traded_volume_df.index
half_life = 3600
model_type = "linear"
cum_impact = impact_state(traded_volume_df, monthly_scaling_factor, half_life, model_type)

In [24]:
def fit_impact_model(
    bin_df: pd.DataFrame,
    scaling_df: pd.DataFrame,
    model_type: str = "linear",
    half_life: int = 3600,
    horizon: int = 6,
    in_sample_month: int | None = None,
    min_time: str = "10:00:00",
) -> pd.DataFrame:
    """
    Fit a discretized OW-style price impact model.

    Model:
        future_return ≈ alpha + lambda * change_in_impact_state

    Parameters
    ----------
    bin_df:
        Bin sample dataframe with columns:
        stock, date, time, orderFlow, mid

    scaling_df:
        Dataframe with columns:
        stock, date, volume, px_vol

    model_type:
        "linear" or "sqrt"

    half_life:
        Impact half-life in seconds.

    horizon:
        Number of bins ahead for returns.
        If bins are 10 seconds, horizon=6 means 1 minute.

    in_sample_month:
        If given, fit on this month and evaluate on next month.
        If None, fit on all data.

    min_time:
        Ignore early open period before this time.

    Returns
    -------
    DataFrame with stock-level fitted parameters:
        beta_estimate = lambda
        alpha_estimate
        is_rsq
        oos_rsq, if in_sample_month is provided
    """

    if model_type not in {"linear", "sqrt"}:
        raise ValueError("model_type must be either 'linear' or 'sqrt'")

    df = bin_df.copy()
    sc = scaling_df.copy()

    df["date"] = pd.to_datetime(df["date"])
    sc["date"] = pd.to_datetime(sc["date"])

    # --------------------------------------------------
    # 1. Build order-flow matrix: rows = (stock, date), columns = time
    # --------------------------------------------------
    traded_volume_df = (
        df.pivot_table(
            index=["stock", "date"],
            columns="time",
            values="orderFlow",
            aggfunc="sum",
        )
        .sort_index()
        .sort_index(axis=1)
        .fillna(0.0)
    )

    # --------------------------------------------------
    # 2. Build price matrix: rows = (stock, date), columns = time
    # --------------------------------------------------
    px_df = (
        df.pivot_table(
            index=["stock", "date"],
            columns="time",
            values="mid",
            aggfunc="last",
        )
        .sort_index()
        .sort_index(axis=1)
    )

    px_df = px_df.ffill(axis=1).bfill(axis=1)

    # --------------------------------------------------
    # 3. Align scaling factors
    # --------------------------------------------------
    monthly_scaling_factor = (
        sc.set_index(["stock", "date"])
        .loc[traded_volume_df.index, ["volume", "px_vol"]]
    )

    # --------------------------------------------------
    # 4. Compute OW impact state
    # --------------------------------------------------
    if model_type == "linear":
        kernel = lambda x: x
    else:
        kernel = lambda x: np.sign(x) * np.sqrt(np.abs(x))

    beta = np.log(2) / (half_life / 10)
    decay_factor = np.exp(-beta)

    pre_ewm = traded_volume_df.copy()

    # Normalize order flow by ADV
    pre_ewm = pre_ewm.divide(monthly_scaling_factor["volume"], axis="rows")

    # Apply impact kernel
    pre_ewm = kernel(pre_ewm)

    # Scale by price volatility
    pre_ewm = pre_ewm.multiply(monthly_scaling_factor["px_vol"], axis="rows")

    # OW normalization used in notebook
    pre_ewm.iloc[:, 1:] /= (1 - decay_factor)

    # Exponentially decayed cumulative impact state
    cum_impact = (
        pre_ewm.T
        .ewm(alpha=1 - decay_factor, adjust=False)
        .mean()
        .T
    )

    # --------------------------------------------------
    # 5. Build regression variables
    # x = change in impact state
    # y = future price return
    # --------------------------------------------------
    x = (
        cum_impact
        .diff(horizon, axis="columns")
        .T
        .unstack()
        .reset_index()
        .rename(columns={"level_2": "time", 0: "x"})
    )

    y = (
        px_df
        .pct_change(horizon, axis="columns")
        .T
        .unstack()
        .reset_index()
        .rename(columns={"level_2": "time", 0: "y"})
    )

    reg_df = x.copy()
    reg_df["y"] = y["y"]

    reg_df = reg_df.loc[reg_df["time"] >= min_time].dropna().copy()

    reg_df["xy"] = reg_df["x"] * reg_df["y"]
    reg_df["xx"] = reg_df["x"] ** 2
    reg_df["yy"] = reg_df["y"] ** 2
    reg_df["count"] = 1

    # Daily stock-level sufficient statistics
    daily_stats = (
        reg_df
        .groupby(["stock", "date"])[["xy", "xx", "yy", "x", "y", "count"]]
        .sum()
        .reset_index()
    )

    # Optional filter from notebook: remove extremely low-return days
    daily_stats = daily_stats.loc[daily_stats["yy"] >= 0.0001].copy()

    # --------------------------------------------------
    # 6. If no train/test month, fit on all data
    # --------------------------------------------------
    if in_sample_month is None:
        summary = daily_stats.groupby("stock")[["xy", "xx", "yy", "x", "y", "count"]].sum()

        denom = summary["xx"] - summary["x"] ** 2 / summary["count"]

        summary["beta_estimate"] = (
            summary["xy"] - summary["x"] * summary["y"] / summary["count"]
        ) / denom

        summary["alpha_estimate"] = (
            summary["y"] / summary["count"]
            - summary["beta_estimate"] * summary["x"] / summary["count"]
        )

        summary["sse"] = summary["yy"] - summary["y"] ** 2 / summary["count"]

        summary["mse"] = (
            summary["yy"]
            - 2 * summary["beta_estimate"] * summary["xy"]
            - 2 * summary["alpha_estimate"] * summary["y"]
            + 2 * summary["alpha_estimate"] * summary["beta_estimate"] * summary["x"]
            + summary["beta_estimate"] ** 2 * summary["xx"]
            + summary["alpha_estimate"] ** 2 * summary["count"]
        )

        summary["is_rsq"] = 1 - summary["mse"] / summary["sse"]

        return (
            summary[["beta_estimate", "alpha_estimate", "is_rsq"]]
            .reset_index()
            .rename(columns={"beta_estimate": "lambda_estimate"})
        )

    # --------------------------------------------------
    # 7. Train on month t, test on month t+1
    # --------------------------------------------------
    daily_stats["date"] = pd.to_datetime(daily_stats["date"])

    is_df = daily_stats.loc[daily_stats["date"].dt.month == in_sample_month]
    oos_df = daily_stats.loc[daily_stats["date"].dt.month == in_sample_month + 1]

    is_summary = is_df.groupby("stock")[["xy", "xx", "yy", "x", "y", "count"]].sum()
    oos_summary = oos_df.groupby("stock")[["xy", "xx", "yy", "x", "y", "count"]].sum()

    is_summary.columns = "is_" + is_summary.columns
    oos_summary.columns = "oos_" + oos_summary.columns

    summary = pd.merge(
        is_summary,
        oos_summary,
        left_index=True,
        right_index=True,
        how="inner",
    )

    denom = summary["is_xx"] - summary["is_x"] ** 2 / summary["is_count"]

    summary["beta_estimate"] = (
        summary["is_xy"] - summary["is_x"] * summary["is_y"] / summary["is_count"]
    ) / denom

    summary["alpha_estimate"] = (
        summary["is_y"] / summary["is_count"]
        - summary["beta_estimate"] * summary["is_x"] / summary["is_count"]
    )

    summary["is_sse"] = summary["is_yy"] - summary["is_y"] ** 2 / summary["is_count"]

    summary["is_mse"] = (
        summary["is_yy"]
        - 2 * summary["beta_estimate"] * summary["is_xy"]
        - 2 * summary["alpha_estimate"] * summary["is_y"]
        + 2 * summary["alpha_estimate"] * summary["beta_estimate"] * summary["is_x"]
        + summary["beta_estimate"] ** 2 * summary["is_xx"]
        + summary["alpha_estimate"] ** 2 * summary["is_count"]
    )

    summary["is_rsq"] = 1 - summary["is_mse"] / summary["is_sse"]

    summary["oos_sse"] = summary["oos_yy"] - summary["oos_y"] ** 2 / summary["oos_count"]

    summary["oos_mse"] = (
        summary["oos_yy"]
        - 2 * summary["beta_estimate"] * summary["oos_xy"]
        - 2 * summary["alpha_estimate"] * summary["oos_y"]
        + 2 * summary["alpha_estimate"] * summary["beta_estimate"] * summary["oos_x"]
        + summary["beta_estimate"] ** 2 * summary["oos_xx"]
        + summary["alpha_estimate"] ** 2 * summary["oos_count"]
    )

    summary["oos_rsq"] = 1 - summary["oos_mse"] / summary["oos_sse"]

    return (
        summary[["beta_estimate", "alpha_estimate", "is_rsq", "oos_rsq"]]
        .reset_index()
        .rename(columns={"beta_estimate": "lambda_estimate"})
    )